In [1]:
# ==========================================================
# 🚀 CNN + TM ABLATION (ALL VARIANTS + CSV LOGGING)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit

# ================= CONFIG =================

BASE_DIR = "MSTAR-10-Classes"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 32
CNN_EPOCHS = 40
TM_EPOCHS = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLAUSES = 4000
T = 2500
S = 6.0

# ================= DATASET =================

import gc
import torch

def clear_cache():
    gc.collect()  # clear python memory

    if torch.cuda.is_available():
        torch.cuda.empty_cache()   # free unused GPU memory
        torch.cuda.ipc_collect()   # clean inter-process memory

    try:
        import pycuda.driver as cuda
        cuda.Context.synchronize()
    except:
        pass

    print("✅ Cache cleared")

class SARDataset(Dataset):
    def __init__(self, folder, limit=None):
        self.samples = []
        classes = sorted(os.listdir(folder))

        for label, cname in enumerate(classes):
            paths = glob(os.path.join(folder, cname, "*"))

            if limit:
                paths = random.sample(paths, min(limit, len(paths)))

            for p in paths:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.stack([img]*3, axis=0)

        return torch.tensor(img), torch.tensor(label)

# ================= LOAD =================

train_dataset = SARDataset(TRAIN_DIR, TRAIN_PER_CLASS)
test_dataset  = SARDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_classes = len(os.listdir(TRAIN_DIR))

# ==========================================================
# 🔧 FLEXIBLE CNN
# ==========================================================

class CNN(nn.Module):
    def __init__(self, use3, use5, use7):
        super().__init__()

        self.use3 = use3
        self.use5 = use5
        self.use7 = use7

        if use3: self.conv3 = nn.Conv2d(3,16,3,padding=1)
        if use5: self.conv5 = nn.Conv2d(3,16,5,padding=2)
        if use7: self.conv7 = nn.Conv2d(3,16,7,padding=3)

        in_channels = 16 * (use3 + use5 + use7)

        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        feats = []
        if self.use3: feats.append(F.relu(self.conv3(x)))
        if self.use5: feats.append(F.relu(self.conv5(x)))
        if self.use7: feats.append(F.relu(self.conv7(x)))

        x = torch.cat(feats, dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

# ==========================================================
# 🔥 TRAIN CNN + LOG
# ==========================================================

def train_cnn(name, use3, use5, use7):
    model = CNN(use3, use5, use7).to(DEVICE)
    classifier = nn.Linear(256, num_classes).to(DEVICE)

    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(classifier.parameters()),
        lr=0.0003
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    history = []

    for epoch in range(CNN_EPOCHS):
        model.train()
        classifier.train()

        for x,y in train_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            feat = model(x)
            pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
            out = classifier(pooled)

            loss = criterion(out,y)
            loss.backward()
            optimizer.step()

        # eval
        model.eval()
        classifier.eval()

        preds_all, labels_all = [], []

        with torch.no_grad():
            for x,y in test_loader:
                x = x.to(DEVICE)
                feat = model(x)
                pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
                out = classifier(pooled)

                preds = torch.argmax(out, dim=1).cpu().numpy()
                preds_all.extend(preds)
                labels_all.extend(y.numpy())

        acc = accuracy_score(labels_all, preds_all)
        history.append([epoch+1, acc])
        print(f"{name} Epoch {epoch+1}: {acc:.4f}")

    pd.DataFrame(history, columns=["epoch","accuracy"]).to_csv(f"{name}.csv", index=False)

    return model, acc

# ==========================================================
# 🔥 FEATURE EXTRACTION FOR TM
# ==========================================================

def extract(loader, model):
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            pooled = F.adaptive_avg_pool2d(feat,1).view(xb.size(0), -1)
            X.append((pooled > 0).cpu().numpy().astype(np.uint32))
            y.extend(yb.numpy())

    return np.vstack(X), np.array(y, dtype=np.uint32)

# ==========================================================
# 🔥 RUN ALL EXPERIMENTS
# ==========================================================

configs = {
    "cnn_3x3": (True, False, False),
    "cnn_5x5": (False, True, False),
    "cnn_7x7": (False, False, True),
    "cnn_3x3_5x5": (True, True, False),
    "cnn_3x3_5x5_7x7": (True, True, True)
}

results = {}

for name, (u3,u5,u7) in configs.items():
    print(f"\nRunning {name}")

    model, acc = train_cnn(name, u3, u5, u7)
    results[name] = acc

    # ===== TM =====
    print(f"Running TM for {name}")

    X_train, y_train = extract(train_loader, model)
    X_test, y_test   = extract(test_loader, model)

    tm = TMClassifier(
        number_of_clauses=CLAUSES,
        T=T,
        s=S,
        weighted_clauses=True,
        feature_negation=True,
        platform="CUDA"
    )

    history_tm = []

    for epoch in range(TM_EPOCHS):
        tm.fit(X_train, y_train, epochs=1)
        preds = tm.predict(X_test)
        acc_tm = accuracy_score(y_test, preds)

        history_tm.append([epoch+1, acc_tm])

    pd.DataFrame(history_tm, columns=["epoch","accuracy"])\
        .to_csv(f"{name}_tm.csv", index=False)

    results[name+"_tm"] = acc_tm

    # =========================
    # 🔥 CLEAR CACHE HERE
    # =========================
    del model
    del tm
    del X_train, X_test, y_train, y_test

    clear_cache()
# ==========================================================
# 📊 FINAL RESULTS
# ==========================================================

print("\n=== FINAL TABLE ===")

for k,v in results.items():
    print(f"{k:25s}: {v:.4f}")


Running cnn_3x3
cnn_3x3 Epoch 1: 0.0808
cnn_3x3 Epoch 2: 0.1200
cnn_3x3 Epoch 3: 0.4400
cnn_3x3 Epoch 4: 0.3052
cnn_3x3 Epoch 5: 0.5159
cnn_3x3 Epoch 6: 0.5192
cnn_3x3 Epoch 7: 0.6598
cnn_3x3 Epoch 8: 0.5695
cnn_3x3 Epoch 9: 0.6726
cnn_3x3 Epoch 10: 0.5171
cnn_3x3 Epoch 11: 0.6425
cnn_3x3 Epoch 12: 0.4227
cnn_3x3 Epoch 13: 0.6429
cnn_3x3 Epoch 14: 0.7324
cnn_3x3 Epoch 15: 0.7357
cnn_3x3 Epoch 16: 0.5769
cnn_3x3 Epoch 17: 0.8161
cnn_3x3 Epoch 18: 0.7765
cnn_3x3 Epoch 19: 0.6252
cnn_3x3 Epoch 20: 0.5241
cnn_3x3 Epoch 21: 0.8355
cnn_3x3 Epoch 22: 0.6751
cnn_3x3 Epoch 23: 0.5897
cnn_3x3 Epoch 24: 0.8449
cnn_3x3 Epoch 25: 0.7390
cnn_3x3 Epoch 26: 0.7922
cnn_3x3 Epoch 27: 0.6746
cnn_3x3 Epoch 28: 0.8445
cnn_3x3 Epoch 29: 0.7992
cnn_3x3 Epoch 30: 0.8561
cnn_3x3 Epoch 31: 0.8054
cnn_3x3 Epoch 32: 0.8911
cnn_3x3 Epoch 33: 0.9085
cnn_3x3 Epoch 34: 0.7212
cnn_3x3 Epoch 35: 0.7806
cnn_3x3 Epoch 36: 0.8256
cnn_3x3 Epoch 37: 0.9340
cnn_3x3 Epoch 38: 0.7588
cnn_3x3 Epoch 39: 0.6973
cnn_3x3 Epoch 40:

In [2]:
# ==========================================================
# 🚀 PATCH ABLATION (1x1 vs 2x2 vs 4x4)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import gc

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit

# ================= CONFIG =================

BASE_DIR = "MSTAR-10-Classes"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 32
CNN_EPOCHS = 40
TM_EPOCHS = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLAUSES = 4000
T = 2500
S = 6.0

# ================= DATASET =================

class SARDataset(Dataset):
    def __init__(self, folder, limit=None):
        self.samples = []
        classes = sorted(os.listdir(folder))

        for label, cname in enumerate(classes):
            paths = glob(os.path.join(folder, cname, "*"))

            if limit:
                paths = random.sample(paths, min(limit, len(paths)))

            for p in paths:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.stack([img]*3, axis=0)

        return torch.tensor(img), torch.tensor(label)

# ================= LOAD =================

train_dataset = SARDataset(TRAIN_DIR, TRAIN_PER_CLASS)
test_dataset  = SARDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_classes = len(os.listdir(TRAIN_DIR))

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

# ================= TRAIN CNN =================

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ==========================================================
# 🔥 PATCH-BASED FEATURE EXTRACTION
# ==========================================================

def extract(loader, PATCH_GRID):
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    h, w = ch_np.shape
                    patch_h = h // PATCH_GRID
                    patch_w = w // PATCH_GRID

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):

                            patch = ch_np[
                                i*patch_h:(i+1)*patch_h,
                                j*patch_w:(j+1)*patch_w
                            ]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std  / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                batch_feats.append(sample_features)

            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

# ==========================================================
# 🔥 PATCH ABLATION LOOP
# ==========================================================

patch_sizes = [1, 2, 4]
final_results = {}

for p in patch_sizes:
    print(f"\n=== Running Patch {p}x{p} ===")

    X_train, y_train = extract(train_loader, p)
    X_test, y_test   = extract(test_loader, p)

    tm = TMClassifier(
        number_of_clauses=CLAUSES,
        T=T,
        s=S,
        weighted_clauses=True,
        feature_negation=True,
        platform="CUDA"
    )

    history = []

    for epoch in range(TM_EPOCHS):
        tm.fit(X_train, y_train, epochs=1)
        preds = tm.predict(X_test)
        acc = accuracy_score(y_test, preds)

        history.append([epoch+1, acc])
        print(f"Patch {p} Epoch {epoch+1}: {acc:.4f}")

    # SAVE CSV
    df = pd.DataFrame(history, columns=["epoch","accuracy"])
    df.to_csv(f"patch_{p}x{p}.csv", index=False)

    final_results[p] = acc

    # CLEAR MEMORY
    del tm, X_train, X_test, y_train, y_test
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================================
# 📊 FINAL RESULTS
# ==========================================================

print("\n=== PATCH ABLATION RESULTS ===")
for p, acc in final_results.items():
    print(f"{p}x{p} → {acc:.4f}")

Training CNN...
✅ CNN Done

=== Running Patch 1x1 ===
2026-04-13 23:41:35,484 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-13 23:41:35,486 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-13 23:41:35,487 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-13 23:41:35,488 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-13 23:41:35,490 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-13 23:41:35,919 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kerne

In [2]:
# ==========================================================
# 🚀 CNN → BINARIZATION ABLATION → TM → EXCEL (FINAL)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit
import pandas as pd

# ================= CONFIG =================

BASE_DIR = "MSTAR-10-Classes"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 32
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ================= EXPERIMENTS =================

EXPERIMENTS = [
    ("Mean only",   "mean",        "3bit"),
    ("Std only",    "std",         "3bit"),
    ("CV only",     "cv",          "3bit"),
    ("Max only",    "max",         "3bit"),
    ("Full",        "full",        "1bit"),
    ("Full",        "full",        "3bit"),
    ("Full",        "full",        "5bit"),
    ("Full",        "full",        "percentile"),
    ("No-relative", "no_relative", "3bit"),
]

# ================= DATASET =================

class SARDataset(Dataset):
    def __init__(self, folder, limit=None):
        self.samples = []
        classes = sorted(os.listdir(folder))

        for label, cname in enumerate(classes):
            paths = glob(os.path.join(folder, cname, "*"))

            if limit:
                paths = random.sample(paths, min(limit, len(paths)))

            for p in paths:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        img = np.stack([img]*3, axis=0)

        return torch.tensor(img), torch.tensor(label)

# ================= LOAD =================

train_dataset = SARDataset(TRAIN_DIR, TRAIN_PER_CLASS)
test_dataset  = SARDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_classes = len(os.listdir(TRAIN_DIR))

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= BINARIZATION =================

def get_thresholds(values, mode):
    if mode == "1bit":
        return [1.0]
    elif mode == "3bit":
        return [0.8, 1.0, 1.2]
    elif mode == "5bit":
        return [0.6, 0.8, 1.0, 1.2, 1.4]
    elif mode == "percentile":
        return np.percentile(values, [25, 50, 75])


def binarize(value, thresholds):
    return [int(value > t) for t in thresholds]


def build_features(mean, std, mx, ch_mean, ch_std, mode_f, mode_t):

    mean_rel = mean / (ch_mean + 1e-6)
    std_rel  = std  / (ch_std + 1e-6)
    cv       = std / (mean + 1e-6)

    feats = []

    if mode_t == "percentile":
        TH_mean = get_thresholds([mean_rel], "percentile")
        TH_std  = get_thresholds([std_rel], "percentile")
    else:
        TH_mean = get_thresholds([mean_rel], mode_t)
        TH_std  = get_thresholds([std_rel], mode_t)

    if mode_f == "mean":
        feats += binarize(mean_rel, TH_mean)

    elif mode_f == "std":
        feats += binarize(std_rel, TH_std)

    elif mode_f == "cv":
        feats += binarize(cv, [0.5, 1.0])

    elif mode_f == "max":
        feats += [int(mx > ch_mean), int(mx > ch_mean + ch_std)]

    elif mode_f == "mean_std":
        feats += binarize(mean_rel, TH_mean)
        feats += binarize(std_rel, TH_std)

    elif mode_f == "no_relative":
        feats += binarize(mean, [0.2, 0.4, 0.6])
        feats += binarize(std, [0.1, 0.2])

    elif mode_f == "full":
        feats += binarize(mean_rel, TH_mean)
        feats += binarize(std_rel, TH_std)
        feats += binarize(cv, [0.5, 1.0])
        feats += [int(mx > ch_mean), int(mx > ch_mean + ch_std)]

    return feats

# ================= FEATURE EXTRACTION =================

def extract(loader, mode_f, mode_t):
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH = 4
                    h, w = ch_np.shape
                    ph, pw = h // PATCH, w // PATCH

                    for i in range(PATCH):
                        for j in range(PATCH):

                            patch = ch_np[i*ph:(i+1)*ph, j*pw:(j+1)*pw]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            f = build_features(mean, std, mx, ch_mean, ch_std, mode_f, mode_t)
                            sample_features.extend(f)

                batch_feats.append(sample_features)

            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

# ================= ABLATION =================

writer = pd.ExcelWriter("binarization_ablation_final.xlsx", engine='xlsxwriter')

summary_results = []

print("\n🚀 STARTING ABLATION...\n")

for exp_name, mode_f, mode_t in EXPERIMENTS:

    print(f"\n=== {exp_name} | {mode_t} ===")

    X_train, y_train = extract(train_loader, mode_f, mode_t)
    X_test, y_test   = extract(test_loader, mode_f, mode_t)

    tm = TMClassifier(
        number_of_clauses=CLAUSES,
        T=T,
        s=S,
        weighted_clauses=True,
        feature_negation=True,
        platform="CPU"
    )

    epoch_logs = []
    best_acc = 0

    for epoch in range(TM_EPOCHS):
        tm.fit(X_train, y_train, epochs=1)
        preds = tm.predict(X_test)
        acc = accuracy_score(y_test, preds)

        epoch_logs.append([epoch+1, acc])
        best_acc = max(best_acc, acc)

        print(f"{exp_name}-{mode_t} → Epoch {epoch+1}: {acc:.4f}")

    # Save per experiment
    df_epoch = pd.DataFrame(epoch_logs, columns=["Epoch", "Accuracy"])
    df_epoch.to_excel(writer, sheet_name=(exp_name + "_" + mode_t)[:30], index=False)

    summary_results.append([exp_name, mode_f, mode_t, best_acc])

# ================= SAVE SUMMARY =================

df_summary = pd.DataFrame(summary_results, columns=[
    "Method", "Feature Set", "Thresholds", "Accuracy"
])

df_summary.to_excel(writer, sheet_name="Summary", index=False)

writer.close()

print("\n✅ DONE! Results saved to Excel.")

Training CNN...
✅ CNN Done

🚀 STARTING ABLATION...


=== Mean only | 3bit ===
Mean only-3bit → Epoch 1: 0.7715
Mean only-3bit → Epoch 2: 0.9728
Mean only-3bit → Epoch 3: 0.9682
Mean only-3bit → Epoch 4: 0.9753
Mean only-3bit → Epoch 5: 0.9765
Mean only-3bit → Epoch 6: 0.9777
Mean only-3bit → Epoch 7: 0.9802
Mean only-3bit → Epoch 8: 0.9794
Mean only-3bit → Epoch 9: 0.9794
Mean only-3bit → Epoch 10: 0.9794
Mean only-3bit → Epoch 11: 0.9827
Mean only-3bit → Epoch 12: 0.9814
Mean only-3bit → Epoch 13: 0.9806
Mean only-3bit → Epoch 14: 0.9806
Mean only-3bit → Epoch 15: 0.9819
Mean only-3bit → Epoch 16: 0.9823
Mean only-3bit → Epoch 17: 0.9819
Mean only-3bit → Epoch 18: 0.9823
Mean only-3bit → Epoch 19: 0.9819
Mean only-3bit → Epoch 20: 0.9802
Mean only-3bit → Epoch 21: 0.9810
Mean only-3bit → Epoch 22: 0.9823
Mean only-3bit → Epoch 23: 0.9794
Mean only-3bit → Epoch 24: 0.9802
Mean only-3bit → Epoch 25: 0.9802
Mean only-3bit → Epoch 26: 0.9810
Mean only-3bit → Epoch 27: 0.9814
Mean only-3bi

In [1]:
# ==========================================================
# 🚀 SAR → CNN → STANDARD BINARIZER → TM + INTERPRETATION
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
from tmu.preprocessing.standard_binarizer.binarizer import StandardBinarizer

# ================= CONFIG =================

BASE_DIR = "MSTAR-10-Classes"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 32
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ================= DATASET =================

class SARDataset(Dataset):
    def __init__(self, folder, limit=None):
        self.samples = []
        classes = sorted(os.listdir(folder))

        for label, cname in enumerate(classes):
            paths = glob(os.path.join(folder, cname, "*"))

            if limit:
                paths = random.sample(paths, min(limit, len(paths)))

            for p in paths:
                self.samples.append((p, label))

    def __len__(self): 
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        img = np.stack([img]*3, axis=0)
        return torch.tensor(img), torch.tensor(label)

# ================= LOAD =================

train_dataset = SARDataset(TRAIN_DIR, TRAIN_PER_CLASS)
test_dataset  = SARDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_classes = len(os.listdir(TRAIN_DIR))

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

def extract(loader):
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)

            feat = model(xb)
            pooled = F.adaptive_avg_pool2d(feat, 1)
            pooled = pooled.view(pooled.size(0), -1)

            X.append(pooled.cpu().numpy())
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.float32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test   = extract(test_loader)

# ================= STANDARD BINARIZER =================

print("Applying Standard Binarizer...")

binarizer = StandardBinarizer(max_bits_per_feature=3)
binarizer.fit(X_train)

X_train = binarizer.transform(X_train).astype(np.uint32)
X_test  = binarizer.transform(X_test).astype(np.uint32)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CPU"
)

print("Training TM...")

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)
    preds = tm.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Epoch {epoch+1} → {acc:.4f}")

print("🔥 FINAL ACC:", acc)

# ==========================================================
# 🔥 INTERPRETATION (CORRECT TMU)
# ==========================================================

def generate_tm_heatmap(idx=0, save_dir="outputs_tm"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_dataset[idx]
    img_np = img.mean(dim=0).numpy()

    x = X_test[idx].reshape(1, -1)

    pred = tm.predict(x)[0]

    # reshape transform output
    clause_outputs_all = tm.transform(x).reshape(
        1, num_classes, tm.number_of_clauses
    )
    clause_outputs = clause_outputs_all[0, pred]

    # get weights
    weights = tm.weight_banks[pred].get_weights()

    feature_dim = 256
    bits = X_test.shape[1] // feature_dim

    channel_importance = np.zeros(feature_dim)

    for cid in range(tm.number_of_clauses):

        if clause_outputs[cid] == 0:
            continue

        w = weights[cid]
        if w == 0:
            continue

        try:
            literals = tm.clause_banks[pred].get_literals(cid)
        except:
            literals = tm.clause_banks[pred].get_literals(cid, 0)
        
        # 🔥 safety handling
        if literals is None:
            continue
        
        literals = np.array(literals).flatten()
        
        for lit in literals:
        
            try:
                lit = int(lit)
            except:
                continue
        
            if lit >= X_test.shape[1]:
                fid = lit - X_test.shape[1]
                sign = -1
            else:
                fid = lit
                sign = 1
        
            ch = fid // bits
        
            if ch < feature_dim:   # 🔥 prevent overflow
                channel_importance[ch] += w * sign

    # normalize
    if np.max(np.abs(channel_importance)) > 0:
        channel_importance /= np.max(np.abs(channel_importance))

    # ⚠️ global heatmap (no spatial info)
    heatmap = np.zeros_like(img_np)
    heatmap += np.sum(channel_importance)

    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)
    heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img_uint8 = (img_np * 255).astype(np.uint8)
    img_color = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(img_color, 0.6, heatmap_color, 0.4, 0)

    cv2.imwrite(f"{save_dir}/heatmap_{idx}.png", heatmap_color)
    cv2.imwrite(f"{save_dir}/overlay_{idx}.png", overlay)

    print(f"✅ Saved sample {idx}")

# ================= RUN =================

print("Generating TM interpretation...")

for i in range(5):
    generate_tm_heatmap(i)

Training CNN...
✅ CNN Done
Extracting features...
Applying Standard Binarizer...
Training TM...
Epoch 1 → 0.9695
Epoch 2 → 0.9806
Epoch 3 → 0.9802
Epoch 4 → 0.9761
Epoch 5 → 0.9798
Epoch 6 → 0.9798
Epoch 7 → 0.9847
Epoch 8 → 0.9810
Epoch 9 → 0.9843
Epoch 10 → 0.9860
Epoch 11 → 0.9839
Epoch 12 → 0.9827
Epoch 13 → 0.9843
Epoch 14 → 0.9843
Epoch 15 → 0.9856
Epoch 16 → 0.9839
Epoch 17 → 0.9827
Epoch 18 → 0.9827
Epoch 19 → 0.9814
Epoch 20 → 0.9827
Epoch 21 → 0.9819
Epoch 22 → 0.9823
Epoch 23 → 0.9827
Epoch 24 → 0.9831
Epoch 25 → 0.9831
Epoch 26 → 0.9823
Epoch 27 → 0.9819
Epoch 28 → 0.9827
Epoch 29 → 0.9827
Epoch 30 → 0.9827
Epoch 31 → 0.9814
Epoch 32 → 0.9827
Epoch 33 → 0.9827
Epoch 34 → 0.9831
Epoch 35 → 0.9827
Epoch 36 → 0.9827
Epoch 37 → 0.9823
Epoch 38 → 0.9823
Epoch 39 → 0.9823
Epoch 40 → 0.9831
Epoch 41 → 0.9831
Epoch 42 → 0.9831
Epoch 43 → 0.9835
Epoch 44 → 0.9843
Epoch 45 → 0.9827
Epoch 46 → 0.9819
Epoch 47 → 0.9835
Epoch 48 → 0.9823
Epoch 49 → 0.9827
Epoch 50 → 0.9827
🔥 FINAL ACC: 